# Topic 4 — Feature Creation
**Covers:** Polynomial features · Interaction terms · Cross-features · Ratio / domain-driven
feature synthesis · Curse of dimensionality (with a concrete distance-concentration
demonstration) · Feature importance after creation

**Dataset:** `data/raw/feature_creation_data.csv` — PrepEdge student data where the TRUE
underlying relationship includes a genuine **interaction** between `study_hours` and
`attendance_pct` (studying hard while skipping class barely helps, and vice versa) — so
creating the right feature isn't decoration, it recovers a real signal a plain linear model
can't see on its own.

---
## Explanation

Imagine you're making a fruit salad. You have apples and bananas separately — each is fine
alone, but when you CHOP THEM TOGETHER into one bowl, you get a brand new flavor combo that
neither fruit gives you by itself.

- **Interaction term** = the "fruit salad" flavor — a new feature made by MULTIPLYING two
  ingredients together, because their COMBINED effect matters, not just each one alone.
- **Polynomial features** = what if you like an apple, but you like TWO apples even more (not
  just twice as much — like, disproportionately more)? Squaring a number (`x²`) lets a model
  learn that kind of "curved," not-perfectly-straight-line effect.
- **Cross-features** = combining two categories, like "city + course," to make a brand new
  category ("Kota-JEE-students" behave differently than just "Kota students" or "JEE students"
  averaged separately).
- **Ratio / domain-driven feature** = using what YOU know about coaching institutes (not just
  math) to build a smart new column, like "score improvement per hour studied" — a ratio a
  teacher would think of, that a generic algorithm might not invent on its own.
- **Curse of dimensionality** = if you throw EVERY possible fruit combo into your salad bowl —
  apple×banana, apple×banana×orange×grape×mango... — the bowl gets so crowded and confusing
  that you can't tell which combo actually tastes good anymore. More isn't always better!

## 1. WHY — Why create new features at all?

A linear model can only ever draw a straight line (or flat plane) through the data it's given.
If the REAL relationship curves, or depends on how two variables interact, a linear model
literally cannot represent that — no matter how much data you feed it — unless you hand it a
feature that already captures the curve or the interaction.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv("../data/raw/feature_creation_data.csv")
df.head()

,student_id,study_hours,attendance_pct,prev_score,doubt_sessions,final_score
0,3001,7.0,56.5,48.6,1,26.8
1,3002,4.6,68.0,74.6,2,27.8
2,3003,7.5,97.6,67.2,3,26.7
3,3004,6.1,71.0,51.1,6,32.5
4,3005,2.1,47.6,52.6,2,25.7


## 2. Interaction terms — WHY / WHAT / HOW

**Manual illustration.** Two students, same total "effort budget," split differently:

| Student | study_hours | attendance_pct | study_hours + attendance (additive) | study_hours × attendance (interaction) |
|---|---|---|---|---|
| A | 10 | 30 | 40 | 300 |
| B | 5  | 60 | 65 | 300 |

An additive model treats student B as "better prepared" (65 > 40). But the TRUE data-generating
process in our dataset rewards the PRODUCT of study and attendance — both students above
actually have identical interaction value (300), meaning genuinely similar outcomes, something
only the multiplicative (interaction) feature reveals.

In [2]:
df["study_x_attendance"] = df["study_hours"] * df["attendance_pct"]
df[["study_hours", "attendance_pct", "study_x_attendance"]].head()

,study_hours,attendance_pct,study_x_attendance
0,7.0,56.5,395.50
1,4.6,68.0,312.80
2,7.5,97.6,732.00
3,6.1,71.0,433.10
4,2.1,47.6,99.96


In [3]:
# Quantify the value of the interaction term with a quick before/after R^2 comparison
X_no_interact = df[["study_hours", "attendance_pct", "prev_score", "doubt_sessions"]]
X_interact = df[["study_hours", "attendance_pct", "prev_score", "doubt_sessions", "study_x_attendance"]]
y = df["final_score"]

for name, X in [("Without interaction term", X_no_interact), ("With interaction term", X_interact)]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
    model = LinearRegression().fit(Xtr, ytr)
    print(f"{name:26s} R^2 = {r2_score(yte, model.predict(Xte)):.3f}")

Without interaction term   R^2 = 0.382
With interaction term      R^2 = 0.400


The interaction feature recovers real signal a plain linear model otherwise misses entirely —
this is not a marginal trick, it directly reflects how the data was generated.

## 3. Polynomial features — WHY / WHAT / HOW

**Manual illustration.** Suppose true score gain from `doubt_sessions` isn't a straight line
but curves (diminishing or accelerating returns). A degree-2 polynomial adds `x²` as its own
column, letting a LINEAR model fit a CURVE (since it's now linear in `x` and `x²` jointly):

```
doubt_sessions = 3  ->  doubt_sessions^2 = 9
doubt_sessions = 6  ->  doubt_sessions^2 = 36    (quadruple, not double!)
```

In [4]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[["study_hours", "attendance_pct"]])
poly_names = poly.get_feature_names_out(["study_hours", "attendance_pct"])
poly_df = pd.DataFrame(poly_features, columns=poly_names)
poly_df.head()

,study_hours,attendance_pct,study_hours^2,study_hours attendance_pct,attendance_pct^2
0,7.0,56.5,49.00,395.50,3192.25
1,4.6,68.0,21.16,312.80,4624.00
2,7.5,97.6,56.25,732.00,9525.76
3,6.1,71.0,37.21,433.10,5041.00
4,2.1,47.6,4.41,99.96,2265.76


Notice `PolynomialFeatures(degree=2)` automatically generates BOTH the squared terms
(`study_hours^2`, `attendance_pct^2`) AND the interaction term (`study_hours attendance_pct`) —
it's a systematic, automatic way to generate everything we built manually above.

### 3a. Which of the generated polynomial features actually matter?

Creating features is only half the job — we should check whether each new column earns its
place. A quick way: fit a model on the full polynomial set and rank feature importances (or,
for a linear model, look at standardized coefficient magnitudes).

In [5]:
from sklearn.ensemble import RandomForestRegressor

X_poly_full = poly_df.copy()
y = df["final_score"]

rf = RandomForestRegressor(n_estimators=300, random_state=0)
rf.fit(X_poly_full, y)

importance_df = pd.DataFrame({
    "feature": poly_names,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)
importance_df

,feature,importance
3,study_hours attendance_pct,0.521310
1,attendance_pct,0.131845
4,attendance_pct^2,0.128223
2,study_hours^2,0.111867
0,study_hours,0.106754


**What to look for:** the interaction term (`study_hours attendance_pct`) should rank near
the top — matching what we already confirmed directly in Section 2 — while purely squared
terms that don't reflect anything in the true data-generating process typically rank lower.
This is exactly how you'd triage a large, automatically-generated polynomial feature set in
practice: create broadly, then use importance ranking (or the filter/wrapper methods from
Topics 5-6) to keep only what earns its place.

## 4. Cross-features — WHY / WHAT / HOW

**Manual illustration.** Combining two categorical columns into one new joint category. Even
without real category columns in this numeric dataset, the same idea applies to `income_bracket
× course` from Unit 1's dataset: `"<5L_NEET"` behaves differently, as a group, than `<5L`
students averaged across ALL courses, or `NEET` students averaged across all incomes.

In [6]:
# Illustrative cross-feature using binned versions of our numeric columns
study_bin = pd.cut(df["study_hours"], bins=[0, 4, 8, 20], labels=["Low", "Medium", "High"])
attendance_bin = pd.cut(df["attendance_pct"], bins=[0, 60, 85, 100], labels=["Low", "Medium", "High"])

df["study_attendance_cross"] = study_bin.astype(str) + "_" + attendance_bin.astype(str)
df["study_attendance_cross"].value_counts()

study_attendance_cross
Medium_Medium    209
Medium_High      112
High_Medium      101
Low_Medium        85
Medium_Low        60
Low_High          44
High_High         32
High_Low          27
Low_Low           26
Name: count, dtype: int64

In [7]:
# Average final_score per cross-feature group -- reveals combos plain averages hide
df.groupby("study_attendance_cross")["final_score"].mean().sort_values(ascending=False).round(1)

study_attendance_cross
High_High        38.0
High_Medium      35.6
High_Low         32.8
Medium_High      32.8
Medium_Medium    30.9
Medium_Low       28.9
Low_Low          27.7
Low_Medium       26.8
Low_High         26.2
Name: final_score, dtype: float64

Notice the ordering isn't simply "High study, High attendance" beating everything else in a
predictable additive way — some middling combinations outperform expectations while others
underperform, exactly the kind of pattern a plain average-of-two-separate-columns view would
miss, and exactly why cross-features exist.

## 5. Ratio & domain-driven feature synthesis — WHY / WHAT / HOW

**Why:** a domain expert (a teacher, a coaching-institute owner) often knows a meaningful
RATIO or COMBINATION that a generic algorithm has no reason to invent by itself. A few
different flavors of domain-driven features:

In [8]:
# 1) Ratio feature: "improvement efficiency" -- score gained per hour invested
df["score_per_study_hour"] = df["final_score"] / df["study_hours"].replace(0, np.nan)

# 2) Weighted composite index: "engagement index" combining attendance and doubt-clearing
df["engagement_index"] = (df["attendance_pct"] / 100) * (1 + df["doubt_sessions"])

# 3) Relative/comparative feature: how does this student compare to the batch average?
batch_avg_score = df["prev_score"].mean()
df["prev_score_vs_batch_avg"] = df["prev_score"] - batch_avg_score

# 4) Threshold/flag feature (domain rule): "at risk of falling behind" per institute policy
#    (PrepEdge's own rule of thumb: attendance below 60% AND study hours below 4/week)
df["needs_intervention_flag"] = (
    (df["attendance_pct"] < 60) & (df["study_hours"] < 4)
).astype(int)

df[["study_hours", "final_score", "score_per_study_hour", "engagement_index",
    "prev_score", "prev_score_vs_batch_avg", "needs_intervention_flag"]].head(8)

,study_hours,final_score,score_per_study_hour,engagement_index,prev_score,prev_score_vs_batch_avg,needs_intervention_flag
0,7.0,26.8,3.828571,1.130,48.6,-11.352571,0
1,4.6,27.8,6.043478,2.040,74.6,14.647429,0
2,7.5,26.7,3.560000,3.904,67.2,7.247429,0
3,6.1,32.5,5.327869,4.970,51.1,-8.852571,0
4,2.1,25.7,12.238095,1.428,52.6,-7.352571,1
5,8.5,45.5,5.352941,3.480,51.6,-8.352571,0
6,5.8,31.7,5.465517,3.155,64.3,4.347429,0
7,7.5,39.4,5.253333,3.920,73.9,13.947429,0


Each of these encodes something a coaching-institute teacher would immediately recognize as
meaningful — a generic feature-generation tool (like `PolynomialFeatures`) would never
"invent" `needs_intervention_flag`, because it requires knowing the institute's specific
policy threshold, not just the numbers themselves.

## 6. ⚠️ Curse of dimensionality — WHY it's not "just add more features"

**Manual intuition.** Imagine placing 100 points randomly:
- On a 1-D line of length 10 → points are packed closely, easy to find near neighbors.
- On a 2-D square of side 10 → the same 100 points are noticeably sparser.
- On a 10-D "cube" of side 10 → those 100 points are practically alone — almost every point
  is far from every other point. **The volume of space grows exponentially with dimensions,
  but your data doesn't grow with it.**

Generating too many polynomial/interaction/cross-features blows up the feature count, spreads
your fixed number of training rows ever more thinly across that space, and makes it easy for a
model to fit noise (overfitting) rather than signal.

### 6a. A concrete demonstration: distance concentration in high dimensions

A famous, very literal symptom of the curse: as dimensionality grows, the distance between the
CLOSEST pair of random points and the FARTHEST pair of random points converges toward each
other — "near" and "far" stop meaning anything. Let's actually measure this.

In [9]:
from sklearn.metrics import pairwise_distances

rng = np.random.default_rng(0)
n_points = 200

ratios = {}
for n_dims in [1, 2, 5, 10, 50, 100, 500]:
    points = rng.uniform(0, 1, size=(n_points, n_dims))
    dists = pairwise_distances(points)
    np.fill_diagonal(dists, np.nan)
    nearest = np.nanmin(dists, axis=1).mean()
    farthest = np.nanmax(dists, axis=1).mean()
    ratios[n_dims] = farthest / nearest

print(f"{'Dimensions':<12}{'Avg nearest dist':<18}{'Avg farthest dist':<18}{'Farthest/Nearest ratio':<10}")
for n_dims in [1, 2, 5, 10, 50, 100, 500]:
    points = rng.uniform(0, 1, size=(n_points, n_dims))
    dists = pairwise_distances(points)
    np.fill_diagonal(dists, np.nan)
    nearest = np.nanmin(dists, axis=1).mean()
    farthest = np.nanmax(dists, axis=1).mean()
    print(f"{n_dims:<12}{nearest:<18.3f}{farthest:<18.3f}{farthest/nearest:<10.3f}")

Dimensions  Avg nearest dist  Avg farthest dist Farthest/Nearest ratio
1           0.003             0.717             285.954   
2           0.038             0.982             25.934    
5           0.259             1.420             5.480     
10          0.628             1.833             2.918     
50          2.239             3.477             1.553     
100         3.410             4.628             1.357     
500         8.521             9.743             1.143     


**What to look for:** in 1-2 dimensions, the farthest point is MANY times more distant than
the nearest point — "near" and "far" are meaningfully different. As dimensions climb into the
hundreds, that ratio collapses toward 1.0 — every point becomes roughly equidistant from every
other point. Distance-based methods (KNN, K-Means, many clustering/anomaly-detection
techniques) fundamentally rely on "near" and "far" being meaningfully different — this is
precisely why blindly generating hundreds of polynomial/cross features can quietly break them,
even before you consider overfitting.

### 6b. The overfitting symptom, on OUR actual dataset

Now the more familiar version: does adding more and more polynomial complexity on our actual,
small (700-row) dataset help or hurt validated performance?

In [10]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

base_cols = ["study_hours", "attendance_pct", "prev_score", "doubt_sessions"]
X_base = df[base_cols].copy()
y = df["final_score"]

results = {}
for degree in [1, 2, 3, 4, 5]:
    poly_deg = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_deg = poly_deg.fit_transform(X_base)
    scores = cross_val_score(LinearRegression(), X_poly_deg, y, cv=5, scoring="r2")
    results[degree] = (X_poly_deg.shape[1], scores.mean())

print(f"{'Degree':<8}{'#Features':<12}{'Mean CV R^2':<12}")
for degree, (n_feats, mean_r2) in results.items():
    print(f"{degree:<8}{n_feats:<12}{mean_r2:<12.3f}")

Degree  #Features   Mean CV R^2 
1       4           0.372       
2       14          0.365       
3       34          0.349       
4       69          0.222       
5       125         -6.287      


Watch what happens as `degree` increases: the feature count explodes, but cross-validated
`R²` typically **peaks and then falls** — beyond some point, added polynomial complexity
fits training noise rather than real signal. This is the curse of dimensionality showing up as
overfitting, and the distance-concentration effect from Section 6a is the deeper geometric
reason why it happens: with a fixed amount of data spread across ever more dimensions, there's
less and less genuine "closeness" for a model to learn from.

## 7. Recap

- **Interaction terms** capture "the combination matters, not just each part alone."
- **Polynomial features** let a linear model fit curves, not just straight lines — but check
  which generated terms actually earn their place via importance ranking.
- **Cross-features** combine categories into new, more specific groups.
- **Ratio & domain-driven features** encode expert knowledge a generic algorithm wouldn't
  invent (efficiency ratios, comparative/relative features, policy-based flags).
- **Curse of dimensionality**: more engineered features isn't free — distances between points
  stop being meaningful in high dimensions, and beyond a point, added columns fit noise instead
  of signal, especially with limited data. Always validate with cross-validation, not just
  training accuracy.

**ELI5 recap:** chop apple and banana together for a new flavor (interaction); let a curved
line describe "two apples are extra-good" (polynomial); combine city + fruit-type into a new
group (cross-feature); use a grown-up's kitchen wisdom for a smart new recipe (domain-driven);
and don't dump every fruit in the world into one bowl, or you won't be able to tell what's
actually good anymore (curse of dimensionality).